In [1]:
import subprocess, time, os, requests

# STEP 1 - Install zstd first
print("Installing zstd...")
subprocess.run("apt-get install -y zstd", shell=True, check=True)
print("✅ zstd installed!")

# STEP 2 - Now install Ollama
print("\nInstalling Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", 
               shell=True, check=True)
print("✅ Ollama installed!")

# STEP 3 - Start ollama serve
print("\nStarting Ollama server...")
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_HOST"] = "0.0.0.0:11434"

subprocess.Popen("ollama serve", shell=True, env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Waiting for Ollama...")
for i in range(20):
    try:
        r = requests.get("http://localhost:11434", timeout=3)
        print("✅ Ollama is UP:", r.text)
        break
    except:
        print(f"  Waiting... ({i+1}/20)")
        time.sleep(2)
else:
    print("❌ Still not up")
    raise Exception("Ollama failed to start")

# STEP 4 - Pull model
print("\nPulling qwen2.5...")
subprocess.run("ollama pull qwen2.5", shell=True, check=True)
print("✅ Model ready!")

# STEP 5 - Test locally
print("\nTesting...")
res = requests.post("http://localhost:11434/api/generate",
    json={"model": "qwen2.5", "prompt": "Say hello!", "stream": False},
    timeout=120)
print("✅ Response:", res.json()["response"])

Installing zstd...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (505 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
✅ zstd installed!

Installing Ollama...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


✅ Ollama installed!

Starting Ollama server...
Waiting for Ollama...
  Waiting... (1/20)
✅ Ollama is UP: Ollama is running

Pulling qwen2.5...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 2bada8a74506:   1% ▕                  ▏  50 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏ 100 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   4% ▕                  ▏ 202 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 305 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   8% ▕█                 ▏ 359 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  10% ▕█                 ▏ 461 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  12% ▕██                ▏ 565 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  13% ▕██                ▏ 614 MB/4.7 GB                  pulling manifest 
pulling 2bada8a7

✅ Model ready!

Testing...
✅ Response: Hello there! How can I assist you today?


In [6]:
import subprocess, time, requests

NGROK_TOKEN = "3BhodVckskETp9h23PI4x4hcQm0_4sPyceKgWYLMvQNBkQb3L"

# Install ngrok using the correct current URL
subprocess.run(
    "curl -fsSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc > /dev/null && "
    "echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' | tee /etc/apt/sources.list.d/ngrok.list && "
    "apt-get update -q && apt-get install -y ngrok",
    shell=True, check=True
)
print("✅ ngrok installed")

# Authenticate
subprocess.run(f"ngrok authtoken {NGROK_TOKEN}", shell=True, check=True)
print("✅ ngrok authenticated")

# Start tunnel in background
subprocess.Popen(
    "ngrok http 11434 --log=stdout",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait and get URL from ngrok's local API
print("Waiting for tunnel...")
tunnel_url = None
for i in range(20):
    try:
        r = requests.get("http://localhost:4040/api/tunnels", timeout=3)
        tunnels = r.json().get("tunnels", [])
        if tunnels:
            tunnel_url = tunnels[0]["public_url"]
            print("✅ YOUR URL:", tunnel_url)
            break
    except:
        print(f"  Waiting... ({i+1}/20)")
        time.sleep(2)

if not tunnel_url:
    raise Exception("ngrok failed to start")

deb https://ngrok-agent.s3.amazonaws.com buster main
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://ngrok-agent.s3.amazonaws.com buster InRelease [20.3 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,475 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ngrok-agent.s3.amazonaws.com buster/main amd64 Packages [11.6 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 https://r2u.s

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  ngrok
0 upgraded, 1 newly installed, 0 to remove and 153 not upgraded.
Need to get 8,371 kB of archives.
After this operation, 0 B of additional disk space will be used.
Get:1 https://ngrok-agent.s3.amazonaws.com buster/main amd64 ngrok amd64 3.37.3 [8,371 kB]
Fetched 8,371 kB in 0s (33.5 MB/s)
Selecting previously unselected package ngrok.
(Reading database ... 124647 files and directories currently installed.)
Preparing to unpack .../ngrok_3.37.3_amd64.deb ...
Unpacking ngrok (3.37.3) ...
Setting up ngrok (3.37.3) ...
✅ ngrok installed
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ ngrok authenticated
Waiting for tunnel...
  Waiting... (1/20)
✅ YOUR URL: https://6973-35-203-149-73.ngrok-free.app
